In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import scanpy.external as sce
import anndata
import pandas as pd
import numpy as np
import os
from seaborn import clustermap

In [ ]:
epithelial_clustering_step2_outputs = "path/to/your/output/directory"
h5ad_path = epithelial_clustering_step2_outputs + "/epithelium.h5ad"
annotation_outputs = "path/to/your/another/output/directory"

adata = sc.read_h5ad(h5ad_path)
adata

In [ ]:
# Please confirm the following clusters contain epithelial cells.
keep = {"0", "1", "2", "3", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18"}

mask = adata.obs["leiden"].astype(str).isin(keep)

ep = adata[mask].copy()
ep

In [ ]:
#annotation
ep.obs.loc[ep.obs['leiden'] == '0', 'type'] = 'both'
ep.obs.loc[ep.obs['leiden'] == '1', 'type'] = 'both'
ep.obs.loc[ep.obs['leiden'] == '2', 'type'] = 'invasion'
ep.obs.loc[ep.obs['leiden'] == '3', 'type'] = 'lepidic'
#ep.obs.loc[ep.obs['leiden'] == '4', 'type'] = ''
ep.obs.loc[ep.obs['leiden'] == '5', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '6', 'type'] = 'both'
ep.obs.loc[ep.obs['leiden'] == '7', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '8', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '9', 'type'] = 'bronchial_cells'
ep.obs.loc[ep.obs['leiden'] == '10', 'type'] = 'invasion'
ep.obs.loc[ep.obs['leiden'] == '11', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '12', 'type'] = 'pneumocyte'
ep.obs.loc[ep.obs['leiden'] == '13', 'type'] = 'both'
ep.obs.loc[ep.obs['leiden'] == '14', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '15', 'type'] = 'invasion'
ep.obs.loc[ep.obs['leiden'] == '16', 'type'] = 'lepidic'
ep.obs.loc[ep.obs['leiden'] == '17', 'type'] = 'pneumocyte'
ep.obs.loc[ep.obs['leiden'] == '18', 'type'] = 'invasion'

In [ ]:
sc.pl.umap(ep, color=["type"])
sc.pl.umap(ep, color=["batch"])

In [ ]:
plot_name = "UMAP_Sample"

umap = sc.pl.umap(ep, color=['batch'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(annotation_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
plot_name = "UMAP_Leiden"

umap = sc.pl.umap(ep, color=['leiden'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(annotation_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
plot_name = "UMAP_Histology"

umap = sc.pl.umap(ep, color=['type'], title="", show=False)

fig = plt.gcf()
fig.subplots_adjust(top=0.88)
fig.suptitle(plot_name, y=0.95, fontsize=20)

fig.savefig(annotation_outputs + "/" + plot_name + ".tiff", bbox_inches="tight", dpi=800)
plt.close(fig)

In [ ]:
normal_lung = {"0076861__Normal_lung", "0076862__Normal_lung", "0076921__Normal_lung", "0076922__Normal_lung"}

mask = ep.obs["batch"].astype(str).isin(normal_lung)

tumor = ep[~mask].copy()
tumor

In [ ]:
from pathlib import Path
from importlib.metadata import version
import pandas as pd
from IPython.display import display

from xenium_pseudobulk_de import (
    make_pseudobulk,
    fit_pydeseq2,
    plot_results,
)

print('PyDESeq2:', version('pydeseq2'))

In [ ]:
tumor.obs['type'].value_counts(dropna=False)

In [ ]:
SAMPLE_KEY = 'batch'
TYPE_KEY = 'type'
COUNTS_SOURCE = 'counts'
PAIRED = True

MIN_CELLS = 100
MIN_COUNT = 30
MIN_SAMPLES = 3

ALPHA = 0.01
LFC_CUTOFF = 1.0
N_CPUS = 20
N_TOP = 20
SIZE_FACTORS_FIT_TYPE = 'poscounts'
CONTROL_GENES = None
EXCLUDE_GENES = None

OUTDIR = Path(annotation_outputs)

In [ ]:
analysis_adata = tumor

cell_counts = pd.crosstab(
    analysis_adata.obs[SAMPLE_KEY],
    analysis_adata.obs[TYPE_KEY],
    dropna=False,
).reindex(columns=['lepidic', 'invasion', 'pneumocyte'], fill_value=0)

display(cell_counts)


In [ ]:
pb = make_pseudobulk(
    analysis_adata,
    sample_key=SAMPLE_KEY,
    type_key=TYPE_KEY,
    counts_source=COUNTS_SOURCE,
    paired=PAIRED,
    min_cells=MIN_CELLS,
    min_count=MIN_COUNT,
    min_samples=MIN_SAMPLES,
    exclude_genes=EXCLUDE_GENES,
)

display(pb['sample_qc'])
display(pb['gene_qc'].head(10))
display(pb['metadata'])

In [ ]:
meta = pb['metadata']

if PAIRED:
    n_pairs = meta['sample_id'].nunique()
    if n_pairs < 3:
        raise ValueError('less than 3 pairs available')
else:
    n_per_group = meta.groupby('type', observed=True)['sample_id'].nunique().reindex(['lepidic', 'invasion'], fill_value=0)
    display(n_per_group.rename('independent_samples'))

In [ ]:
result = fit_pydeseq2(
    pb,
    outdir=OUTDIR,
    n_cpus=N_CPUS,
    alpha=ALPHA,
    lfc_cutoff=LFC_CUTOFF,
    size_factors_fit_type=SIZE_FACTORS_FIT_TYPE,
    control_genes=CONTROL_GENES,
)

res = result['results']

In [ ]:
figure_files = plot_results(result, n_top=N_TOP, show=True)
figure_files